<a href="https://colab.research.google.com/github/SauravSG/Deep-Learning/blob/main/fine-tuning/QNLI_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **QNLI: fine-tuning BERT to decide whether a sentence answers a question**

## **What QNLI is**
- A sentence-pair task like MRPC, but the two texts have different roles: the first is a **question**, the second is a **sentence** taken from a Wikipedia paragraph.
- The label says whether the sentence contains the answer to the question.
  - `0` = `entailment` (the sentence answers the question)
  - `1` = `not_entailment` (it doesn't)
  - This is the **reverse of MRPC**, where `1` was the positive class.
- The input order matters: `[CLS] question [SEP] sentence [SEP]`. `token_type_ids` are `0` for the question and `1` for the sentence.

**Quick warm up**

In [1]:
# import transformers and torch
from transformers import AutoTokenizer, AutoModel
import torch

# load the tokenizer and the model
model_name = 'bert-base-uncased'
model_v0 = AutoModel.from_pretrained(model_name)
tokenizer_v0 = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [2]:
# define a sample text
sample_text = 'The weather is nice today!'

# tokenize it and convert to pytorch so the model can process
text_output = tokenizer_v0(sample_text, return_tensors='pt')

print(text_output)

{'input_ids': tensor([[ 101, 1996, 4633, 2003, 3835, 2651,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [3]:
# pass the tokenized inputs through the model
with torch.inference_mode():

  output = model_v0(**text_output)

print(output.keys(),'\n')
print(output.last_hidden_state.shape, '\n')
print(output.pooler_output.shape)

odict_keys(['last_hidden_state', 'pooler_output']) 

torch.Size([1, 8, 768]) 

torch.Size([1, 768])


In [4]:
extract_cls_from_output = output.last_hidden_state[0, 0, :]
print(extract_cls_from_output.shape)

torch.Size([768])


In [5]:
print(extract_cls_from_output[:3])

tensor([0.2170, 0.1494, 0.0633])


**Step 1: Load the dataset and look at its structure**

- `load_dataset('nyu-mll/glue', 'qnli')` gives a `DatasetDict`: train 104,743, validation 5,463, test 5,463.
- Columns: `question`, `sentence`, `label`, `idx`.

In [6]:
# import the load_dataset library
from datasets import load_dataset

In [7]:
# load the dataset
raw_qnli = load_dataset(path = 'nyu-mll/glue', name = 'qnli')
print(raw_qnli)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

qnli/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.5MB            

qnli/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

qnli/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  872kB            

qnli/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

qnli/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})


In [8]:
raw_qnli['train']['question'][0]

'When did the third Digimon series begin?'

In [9]:
raw_qnli['train']['sentence'][0]

'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.'

In [10]:
raw_qnli['train']['label'][0]

1

In [11]:
raw_qnli['train'].features

{'question': Value('string'),
 'sentence': Value('string'),
 'label': ClassLabel(names=['entailment', 'not_entailment']),
 'idx': Value('int32')}

**Step 1.1: Check what the label numbers mean**

- `features['label']` is `ClassLabel(names=['entailment', 'not_entailment'])`, so `0` = entailment and `1` = not_entailment.
- Row 0: question "When did the third Digimon series begin?" with a sentence describing the show's tone but not its start date → label `1`.
- The test split's labels are all `-1` (hidden), so it can't be used for evaluation. **Validation is the eval split.**


In [12]:
raw_qnli['train'].features['label'] # 0 = entailment -> the sentence does answer the question, 1 = it doesn't answer the question

ClassLabel(names=['entailment', 'not_entailment'])

In [13]:
set(raw_qnli['train']['label'])

{0, 1}

**Step 2: Count the labels in validation (this gives the baseline floor)**


In [14]:
# count the number of label 0 & 1 from the validation dataset.
from collections import Counter
count = Counter(raw_qnli['validation']['label'])
print(count)

Counter({1: 2761, 0: 2702})


- `Counter` on the validation labels: `{1: 2761, 0: 2702}`, which is almost balanced.
- A model that always says `not_entailment` gets 2761 / 5463 = **50.5%**. This is the floor a trained model has to beat.

**Step 3: Take a random 20,000-pair subsample of train**

- The full 104,743 pairs would be about 6,550 steps per epoch, too slow for free Colab. 20,000 pairs is 1,250 steps per epoch, 3,750 for 3 epochs.
- `shuffle(seed=42)` and then `select(range(20000))`. The seed makes the same 20k come out every time. Validation stays full-size.

In [15]:
# given the training size is 104743, we will consider 20k for training chunk to save us the traning time. We will also seed for the reproducibility instead of picking the first 20k entries in the dataset
small_train_chunk = raw_qnli['train'].shuffle(seed = 42).select(range(20000))

In [16]:
# confirm it's still close to balance

print((small_train_chunk))

Dataset({
    features: ['question', 'sentence', 'label', 'idx'],
    num_rows: 20000
})


**step 3.1: Check the subsample is still balanced**

- For this we will use the `Counter` again - `{1: 10012, 0: 9988}`, so the shuffle didn't skew it.

In [17]:
# confirm it's still close to balance

print(Counter(small_train_chunk['label']))

Counter({1: 10012, 0: 9988})


**step 4: tokenized_function**

- It takes a batch (`example['question']` is a list of strings) and returns the tokenizer output for the pair: `tokenizer(example['question'], example['sentence'], truncation=True)`.
- Question first, sentence second. `truncation=True` cuts texts that are too long.
- No padding here. Padding to one global length wastes compute. It happens later, per batch, in the collator.
- Output fields:
  - `input_ids`: the tokens as numbers
  - `token_type_ids`: `0` for the question part, `1` for the sentence part
  - `attention_mask`: `1` for real tokens, `0` for padding

In [18]:
def tokenized_function(example):

  return tokenizer_v0(example['question'], example['sentence'], truncation=True)

In [19]:
tokenized_function(raw_qnli['train'][:2])

{'input_ids': [[101, 2043, 2106, 1996, 2353, 10667, 16339, 2078, 2186, 4088, 1029, 102, 4406, 1996, 2048, 3692, 2077, 2009, 1998, 2087, 1997, 1996, 3692, 2008, 2628, 1010, 10667, 16339, 2078, 24763, 2869, 3138, 1037, 9904, 1998, 2062, 12689, 3921, 2000, 2049, 2466, 3794, 10667, 16339, 2078, 2040, 2079, 2025, 27788, 10010, 12556, 2044, 2037, 6677, 1998, 2062, 3375, 2839, 2458, 1999, 1996, 2434, 2887, 1012, 102], [101, 2029, 7421, 10274, 2411, 2031, 3265, 22742, 2015, 2195, 3717, 2013, 2028, 2178, 1029, 102, 2043, 2158, 15455, 2015, 2003, 3498, 2011, 15744, 1010, 10274, 2089, 2031, 2195, 6474, 2780, 21296, 2075, 10329, 1999, 2235, 5433, 1025, 2969, 1011, 15801, 2250, 4721, 4409, 2089, 21296, 1999, 7689, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,

**step 5: apply .map()**

- apply mapping to both small_train_chunk and validation dataset and remove the columns questions, sentence and idx as they are no longer needed for the training

- `remove_columns=['question', 'sentence', 'idx']`: the raw text is not needed any more.

In [20]:
# apply .map() on small_train_chunk using the tokenized function to map the input_ids, token_type_ids, attention_mask
train_tok = small_train_chunk.map(tokenized_function, batched = True, remove_columns = ['question', 'sentence', 'idx'])

# applying the .map() on the entire validation dataset as the dataset size is small. We don't need to map the entire train dataset because the size is bigger hence we are mapping separately on small_train_chunk and validation dataset.
validation_tok = raw_qnli['validation'].map(tokenized_function, batched = True, remove_columns = ['question', 'sentence', 'idx'])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

In [21]:
train_tok, validation_tok

(Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 20000
 }),
 Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 5463
 }))

**step 5.1: rename the columns from label to labels**

In [22]:
train_tok = train_tok.rename_column('label', 'labels')
validation_tok = validation_tok.rename_column('label', 'labels')

In [23]:
train_tok, validation_tok

(Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 20000
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 5463
 }))

**step 6: data collator:**

- `DataCollatorWithPadding(tokenizer=...)` pads each batch only up to the longest example **in that batch**. This saves GPU memory and time compared with padding everything to one length.
- `Trainer` calls it automatically. It needs only `input_ids`, `token_type_ids`, `attention_mask`, and `labels`, which is why the extra columns were removed.

In [24]:
# import the necessary library, DataCollatorWithPadding
from transformers import DataCollatorWithPadding

In [25]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer_v0)
print(data_collator)

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')


- For the dynammic padding, it mainly needs list of dictionary that contains `input_ids` , `token_type_ids` and `attention_mask` as these pieces are needed during model training.

**step 7: Define the label names**

In [26]:
# quick exercise
id2label_trial = {0: 'apple', 1:'banana', 2:'orange'}
id2label_trial

{0: 'apple', 1: 'banana', 2: 'orange'}

In [27]:
label2id_trial = {'apple': 0, 'banana': 1, 'orange': 2}
label2id_trial

{'apple': 0, 'banana': 1, 'orange': 2}

In [28]:
id2label_trial.items()

dict_items([(0, 'apple'), (1, 'banana'), (2, 'orange')])

- so below we are going to apply or convert the id2label to label2id in one go and for that we are going to run a loop.

In [29]:
label2id_trial2 = {v:k for k, v in id2label_trial.items()} # we did v:k based on the format we want for label2id and k,v is based on the format the id2label already has
label2id_trial2

{'apple': 0, 'banana': 1, 'orange': 2}

In [30]:
# applying the same learning to the dataset labels
id2label = {0: 'entailment', 1:'not_entailment'}
print(id2label)

# to convert the id2label to label2id, we will use for loop to convert in one go. v:k is because we want in value:key pair for label2id and key,value is for the existing format of id2label
label2id = {v:k for k,v in id2label.items()}
print(label2id)

{0: 'entailment', 1: 'not_entailment'}
{'entailment': 0, 'not_entailment': 1}


- `id2label = {0: 'entailment', 1: 'not_entailment'}`, and `label2id` is the reverse (`{v: k for k, v in id2label.items()}`).
- Without these, the model shows `LABEL_0` and `LABEL_1`.

**step 7.1: Load the model with those names**

Call `set_seed(42)` **first**, because the random classifier head is created inside `from_pretrained`.
- `AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, id2label=id2label, label2id=label2id)`
- The load report is expected: the old MLM and NSP heads are `UNEXPECTED` and are dropped, and the new `classifier` layer is `MISSING` (random, to be trained).


In [31]:
# We need to add the above labeling to the model as well so that it would show the above labels instead of LABLE_0/LABEL_1
from transformers import AutoModelForSequenceClassification
from transformers import set_seed

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2, id2label = id2label, label2id = label2id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
model_name

'bert-base-uncased'

In [33]:
print(model.config.id2label)

{0: 'entailment', 1: 'not_entailment'}


**step 8: get the evaluate lib for evaluation**

- `!pip install evaluate`, then `evaluate.load('glue', 'qnli')`. QNLI returns **accuracy only**.
- Scores are calculated at every evaluation during training (once per epoch here), not continuously.

In [34]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [35]:
import evaluate

metric = evaluate.load("glue", "qnli")
print(metric)

EvaluationModule(name: "glue", module_type: "metric", features: {'predictions': Value('int64'), 'references': Value('int64')}, usage: """
Compute GLUE evaluation metric associated to each GLUE dataset.
Args:
    predictions: list of predictions to score.
        Each translation should be tokenized into a list of tokens.
    references: list of lists of references for each translation.
        Each reference should be tokenized into a list of tokens.
Returns: depending on the GLUE subset, one or several of:
    "accuracy": Accuracy
    "f1": F1 score
    "pearson": Pearson Correlation
    "spearmanr": Spearman Correlation
    "matthews_correlation": Matthew Correlation
Examples:

    >>> glue_metric = evaluate.load('glue', 'sst2')  # 'sst2' or any of ["mnli", "mnli_mismatched", "mnli_matched", "qnli", "rte", "wnli", "hans"]
    >>> references = [0, 1]
    >>> predictions = [0, 1]
    >>> results = glue_metric.compute(predictions=predictions, references=references)
    >>> print(results

**step 8.1: write compute_metrics function to get the evaluation metric (accuracy in this case)**

In [36]:
import numpy as np

def compute_metrics(eval_preds):

  logits, labels = eval_preds.predictions, eval_preds.label_ids

  preds = np.argmax(logits, axis = -1)

  return metric.compute(predictions = preds, references = labels)


**What eval_preds is:**

- `eval_preds` is an `EvalPrediction` object that `Trainer` builds and passes in automatically. You never create it or call `compute_metrics` yourself.
- Each time evaluation runs, `Trainer` runs the model over the whole `eval_dataset` (5,463 pairs), collects the outputs, packs them with the true labels, and calls `compute_metrics`.
- Inside it:
  - `eval_preds.predictions`: the logits, shape `(5463, 2)`. Column 0 is the `entailment` score and column 1 is the `not_entailment` score.
  - `eval_preds.label_ids`: the true labels, shape `(5463,)`.
- Small example: logits `[[2.1, -0.5], [-1.0, 3.0], [0.2, 0.9]]`, labels `[0, 1, 0]` → `argmax` gives `[0, 1, 1]` → 2 of 3 correct → `{'accuracy': 0.667}`.
- The function returns the score dict from `metric.compute(...)`. `preds` and `label_ids` are only its inputs. `Trainer` shows that dict as `eval_accuracy` and uses it to compare checkpoints.


---

**step 9: Create TrainingArguments**

- Only `output_dir` is required. Here it is `qnli-trainer`, a new folder so it doesn't mix with MRPC.
- Settings used: 3 epochs, batch size 16, learning rate 5e-5, `eval_strategy='epoch'`, `save_strategy='epoch'`, `load_best_model_at_end=True`.


In [37]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir = 'qnli-trainer', num_train_epochs = 3, per_device_train_batch_size = 16, learning_rate = 5e-5, eval_strategy = 'epoch',
                                  save_strategy = 'epoch', load_best_model_at_end = True)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

**step 10: Build the `Trainer`**

- Arguments: the model, `args`, `train_dataset` and `eval_dataset`, `data_collator`, `processing_class=tokenizer`, and `compute_metrics`.
- `processing_class` doesn't tokenize anything (the data is already tokenized). It only saves the tokenizer next to the model files so both can be pushed to the Hub.
- `eval_dataset` must be passed here because `eval_strategy` is `'epoch'`.


In [38]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_tok,
    eval_dataset = validation_tok,
    processing_class = tokenizer_v0,  # data is already tokenized; this is only saved with the model checkpoints (needed later for push_to_hub)
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

print(trainer)

- Before training, we will evaluate the 'validation' data to see without the training, what's the baseline score the model gives.

In [39]:
baseline_result = trainer.evaluate()
print(baseline_result)

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.740523,0,0.453780


{'eval_loss': 0.7405232787132263, 'eval_accuracy': 0.4537799743730551}


**step 10.1: train the model using `.train()`**

In [40]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.407296,0.312163,0.872231
2,0.214494,0.363325,0.873696
3,0.098395,0.553670,0.877540


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=3750, training_loss=0.25116703542073565, metrics={'train_runtime': 1275.5941, 'train_samples_per_second': 47.037, 'train_steps_per_second': 2.94, 'total_flos': 2746312084745280.0, 'train_loss': 0.25116703542073565, 'epoch': 3.0})

In [41]:
print(trainer.state.best_model_checkpoint)   # which checkpoint
print(trainer.state.best_metric)             # its score (validation loss here)

qnli-trainer/checkpoint-1250
0.3121632933616638


- Results (validation):
  - Epoch 1: loss 0.312, accuracy 87.2%
  - Epoch 2: loss 0.363, accuracy 87.4%
  - Epoch 3: loss 0.554, accuracy 87.8%
- Training loss kept falling (0.407 → 0.214 → 0.098) while validation loss rose, which means overfitting from epoch 2.



**QNLI run 1 (bert-base-uncased, 20k train subsample, seed 42)**

- Settings: 3 epochs, batch 16, lr 5e-5. Eval on full validation (5,463).
- Floor (always not_entailment): 50.5%. Untrained baseline: 45.4% acc, loss 0.741.
- Hypothesis: val loss lowest at epoch 3, accuracy 70-79%. Result: NOT supported on both counts.
- Epoch 1/2/3: val loss 0.312 / 0.363 / 0.554, accuracy 87.2 / 87.4 / 87.8%.
- Best checkpoint by val loss: epoch 1 (checkpoint-1250). Overfitting from epoch 2.

**step 11: push to HF hub**

- `notebook_login()`, then push `trainer.model` (it holds the best checkpoint) and the tokenizer to a **new** repo (`bert-base-uncased-finetuned-for-qnli`).
- The tokenizer goes along so that anyone can load both with `from_pretrained(repo)` and tokenize raw text the same way the model was trained.
- Then write the model card: the baseline, the floor, the results table, the flipped labels, the 20k subsample, and the note that the validation split was also used to choose the checkpoint.

In [42]:
from huggingface_hub import notebook_login

notebook_login()

In [43]:
repo_name = "bert-base-uncased-finetuned-for-qnli"

In [44]:
# push the trained model
trainer.model.push_to_hub(repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pc4yihl/model.safetensors:   0%|          | 14.2kB /  438MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/GoingDeep/bert-base-uncased-finetuned-for-qnli/commit/3f7efcb2db468fecb6add9923c77b047c9fec97e', commit_message='Upload BertForSequenceClassification', commit_description='', oid='3f7efcb2db468fecb6add9923c77b047c9fec97e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/GoingDeep/bert-base-uncased-finetuned-for-qnli', endpoint='https://huggingface.co', repo_type='model', repo_id='GoingDeep/bert-base-uncased-finetuned-for-qnli'), pr_revision=None, pr_num=None)

In [46]:
# pushed the model's tokenizer that was used in the training
tokenizer_v0.push_to_hub(repo_name)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/GoingDeep/bert-base-uncased-finetuned-for-qnli/commit/736f32028d00580c6f63d9964d7ba17959f0d84f', commit_message='Upload tokenizer', commit_description='', oid='736f32028d00580c6f63d9964d7ba17959f0d84f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/GoingDeep/bert-base-uncased-finetuned-for-qnli', endpoint='https://huggingface.co', repo_type='model', repo_id='GoingDeep/bert-base-uncased-finetuned-for-qnli'), pr_revision=None, pr_num=None)